# Hydrogen Production with NeqSim and Python

This companion notebook gathers the book's practical hydrogen workflows into one place. It covers hydrogen thermodynamics and reference equations, production chemistry and equilibrium, blue-hydrogen route builders, green electrolysis, and transport by compression and pipeline.

Run the cells in a Python environment with the NeqSim repository available. The setup cell loads classes from the workspace so the notebook reflects the same branch used to generate the book.


In [ ]:
import os
import sys
from pathlib import Path


def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    candidates.append(Path.home() / "Documents" / "GitHub" / "neqsim")
    for candidate in candidates:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root. Set NEQSIM_PROJECT_ROOT.")


PROJECT_ROOT = find_neqsim_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

from neqsim_dev_setup import neqsim_init

ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=True)
JClass = ns.JClass

import jpype
J = jpype.JPackage("neqsim")
import math
import json
import pandas as pd
import matplotlib.pyplot as plt

CatalystDeactivationKinetics = JClass("neqsim.process.equipment.reactor.CatalystDeactivationKinetics")
BlueHydrogenPlantBuilder = JClass("neqsim.process.hydrogen.BlueHydrogenPlantBuilder")
PSACascade = JClass("neqsim.process.equipment.adsorber.PSACascade")
PressureSwingAdsorptionBed = JClass("neqsim.process.equipment.adsorber.PressureSwingAdsorptionBed")
PSACostEstimate = JClass("neqsim.process.costestimation.adsorber.PSACostEstimate")
Electrolyzer = JClass("neqsim.process.equipment.electrolyzer.Electrolyzer")
ElectrolyzerTechnology = JClass("neqsim.process.equipment.electrolyzer.ElectrolyzerTechnology")
ElectrolyzerIVCharacteristic = JClass("neqsim.process.equipment.electrolyzer.ElectrolyzerIVCharacteristic")
ElectrolyzerMechanicalDesign = JClass("neqsim.process.mechanicaldesign.electrolyzer.ElectrolyzerMechanicalDesign")
ElectrolyzerCostEstimate = JClass("neqsim.process.costestimation.electrolyzer.ElectrolyzerCostEstimate")
ParaOrthoH2Correction = JClass("neqsim.thermo.util.hydrogen.ParaOrthoH2Correction")

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## 1. Hydrogen Reference Equations and Property Checks

The thermodynamics notebook compares cubic EOS calculations with GERG-2008 mixture properties and Leachman pure-hydrogen properties. Use cubic EOS objects for flowsheets, and use GERG/Leachman calls as reference checks for density, enthalpy, heat capacity, speed of sound, and Joule-Thomson behaviour.


In [ ]:
def h2_blend_system(model, temperature_K, pressure_bara, h2_mole_fraction=0.90):
    system = model(temperature_K, pressure_bara)
    system.addComponent("hydrogen", h2_mole_fraction)
    system.addComponent("methane", 1.0 - h2_mole_fraction)
    system.setMixingRule("classic")
    ops = J.thermodynamicoperations.ThermodynamicOperations(system)
    ops.TPflash()
    system.initProperties()
    return system

temperature_K = 273.15 + 20.0
rows = []
for pressure_bara in [1.01325, 10.0, 100.0, 500.0, 1000.0]:
    srk = h2_blend_system(J.thermo.system.SystemSrkEos, temperature_K, pressure_bara)
    pr = h2_blend_system(J.thermo.system.SystemPrEos, temperature_K, pressure_bara)
    rows.append({
        "pressure_bara": pressure_bara,
        "srk_density_kg_m3": srk.getDensity("kg/m3"),
        "pr_density_kg_m3": pr.getDensity("kg/m3"),
        "gerg_density_kg_m3": srk.getPhase(0).getDensity_GERG2008(),
        "gerg_enthalpy_index_7": srk.getPhase(0).getProperties_GERG2008()[7],
    })

df_ref = pd.DataFrame(rows)
display(df_ref)
ax = df_ref.plot(x="pressure_bara", y=["srk_density_kg_m3", "pr_density_kg_m3", "gerg_density_kg_m3"], marker="o")
ax.set_xlabel("Pressure (bara)")
ax.set_ylabel("Density (kg/m3)")
ax.set_title("H2/methane density: cubic EOS versus GERG-2008 check")
plt.show()


In [ ]:
leachman = J.thermo.system.SystemLeachmanEos(273.15 + 20.0, 90.0)
leachman.addComponent("hydrogen", 1.0)
leachman.init(0)

leachman_summary = {
    "density_normal_kg_m3": leachman.getPhase(0).getDensity_Leachman("normal"),
    "density_para_kg_m3": leachman.getPhase(0).getDensity_Leachman("para"),
    "density_ortho_kg_m3": leachman.getPhase(0).getDensity_Leachman("ortho"),
    "properties_normal": list(leachman.getPhase(0).getProperties_Leachman("normal")),
}
leachman_summary


## 2. Chemical Reaction Equilibrium in NeqSim

The production notebook demonstrates pre-reforming, steam reforming, partial oxidation, and reaction-property ideas. The NeqSim translation uses `GibbsReactor` for equilibrium composition, `StoichiometricReaction` for fixed-conversion balances, and catalyst-screening utilities for activity loss.

Key reactions:

- `CH4 + H2O <-> CO + 3H2`
- `CO + H2O <-> CO2 + H2`
- `CH4 + 1/2 O2 <-> CO + 2H2`
- `CO + 2H2 <-> CH3OH` as a reaction-property teaching example.


In [ ]:
def run_smr_gibbs(temperature_C=850.0, pressure_bara=30.0, steam_to_carbon=3.0):
    fluid = J.thermo.system.SystemSrkEos(273.15 + temperature_C, pressure_bara)
    for name, moles in [
        ("methane", 1.0),
        ("water", steam_to_carbon),
        ("hydrogen", 1.0e-4),
        ("CO", 1.0e-4),
        ("CO2", 1.0e-4),
        ("nitrogen", 0.02),
    ]:
        fluid.addComponent(name, moles)
    fluid.setMixingRule("classic")
    feed = J.process.equipment.stream.Stream("SMR feed", fluid)
    feed.setFlowRate(1000.0, "kg/hr")

    reactor = J.process.equipment.reactor.GibbsReactor("SMR Gibbs reactor", feed)
    reactor.setEnergyMode("isothermal")
    reactor.setMaxIterations(5000)
    reactor.setConvergenceTolerance(1.0e-6)
    reactor.setDampingComposition(0.01)
    reactor.setComponentAsInert("nitrogen")
    reactor.run()

    outlet = reactor.getOutletStream().getThermoSystem()
    in_ch4 = feed.getThermoSystem().getComponent("methane").getNumberOfmoles()
    out_ch4 = outlet.getComponent("methane").getNumberOfmoles()
    return {
        "converged": reactor.hasConverged(),
        "mass_balance_error_pct": reactor.getMassBalanceError(),
        "methane_conversion": (in_ch4 - out_ch4) / in_ch4,
        "reactor_power_MW": reactor.getPower("MW"),
        "h2_mole_fraction": outlet.getPhase(0).getComponent("hydrogen").getz(),
        "co_mole_fraction": outlet.getPhase(0).getComponent("CO").getz(),
        "co2_mole_fraction": outlet.getPhase(0).getComponent("CO2").getz(),
    }

pd.DataFrame([run_smr_gibbs(t) for t in [750.0, 850.0, 950.0]])


In [ ]:
wgs_system = J.thermo.system.SystemSrkEos(273.15 + 350.0, 25.0)
for name, moles in [("CO", 1.0), ("water", 1.5), ("CO2", 1.0e-6), ("hydrogen", 1.0e-6)]:
    wgs_system.addComponent(name, moles)
wgs_system.setMixingRule("classic")

wgs = J.process.equipment.reactor.StoichiometricReaction("fixed WGS conversion")
wgs.addReactant("CO", 1.0)
wgs.addReactant("water", 1.0)
wgs.addProduct("CO2", 1.0)
wgs.addProduct("hydrogen", 1.0)
wgs.setLimitingReactant("CO")
wgs.setConversion(0.85)
reacted_mol = wgs.react(wgs_system)
wgs_system.init(0)
{
    "reacted_CO_mol": reacted_mol,
    "conversion": wgs.getConversion(),
    "hydrogen_moles": wgs_system.getComponent("hydrogen").getNumberOfmoles(),
}


In [ ]:
Kinetics = CatalystDeactivationKinetics
activity = Kinetics(Kinetics.CatalystFamily.NICKEL_REFORMING)
activity.setTemperature(273.15 + 700.0)
activity.setSulfurPpmv(0.05)
activity.setCarbonPotential(0.4)
activity.setSteamToCarbonRatio(2.8)
activity.setOperationHours(8000.0)
{
    "activity_factor": activity.calculateActivity(),
    "dominant_mechanism": str(activity.getDominantMechanism()),
}


## 3. PSA Purification and Cost Hook

Hydrogen equilibrium production is not product production until purification losses are accounted for. The PSA cells estimate cycle-averaged H2 purity, recovery, tail gas, and a Class 4-5 purchased-equipment cost hook.


In [ ]:
syngas = J.thermo.system.SystemSrkEos(313.15, 28.0)
for name, moles in [("hydrogen", 0.72), ("CO2", 0.14), ("methane", 0.04), ("CO", 0.02), ("water", 0.08)]:
    syngas.addComponent(name, moles)
syngas.setMixingRule("classic")
psa_feed = J.process.equipment.stream.Stream("shifted syngas", syngas)
psa_feed.setFlowRate(10000.0, "kg/hr")

psa = PSACascade("H2 PSA", psa_feed)
psa.setConfiguration(PSACascade.CascadeConfiguration.BEDS_8)
psa.setSorbent(PressureSwingAdsorptionBed.SorbentType.ACTIVATED_CARBON)
psa.setPerBedRecoveryTarget(0.85)
psa.run()

psa_cost = PSACostEstimate(psa)
{
    "h2_purity": psa.getH2Purity(),
    "h2_recovery": psa.getH2Recovery(),
    "tail_gas_flow_kg_hr": psa.getTailGasStream().getFlowRate("kg/hr"),
    "purchased_equipment_cost_USD": psa_cost.getPurchasedEquipmentCost(),
}


## 4. Blue Hydrogen Route Builder and CCS Handoff

The blue-hydrogen cells use `BlueHydrogenPlantBuilder` to assemble the screening chain: SMR furnace, HT/LT water-gas shift, cooling and knock-out, selective CO2 capture, CO2 export compression, PSA, H2 drying, and H2 export compression. The capture and dryer units are screening placeholders, so the result is a concept-study boundary rather than a vendor amine or molecular-sieve design.


In [ ]:
blue_builder = BlueHydrogenPlantBuilder()
blue_builder.setName("Notebook blue H2")
blue_builder.setMethaneFeedMolePerSec(120.0)
blue_builder.setSteamToCarbonRatio(3.0)
blue_builder.setCo2CaptureFraction(0.92)
blue_builder.setCo2ExportPressure(110.0)
blue_builder.setH2ExportPressure(100.0)
blue_builder.setIncludePsa(True)

blue_process = blue_builder.build()
blue_process.run()

blue_furnace = blue_builder.getReformerFurnace()
blue_ht_shift = blue_builder.getHighTemperatureShiftReactor()
blue_lt_shift = blue_builder.getLowTemperatureShiftReactor()
blue_capture = blue_builder.getCo2CaptureUnit()
blue_psa = blue_builder.getPsaCascade()

blue_summary = {
    "capture_target_fraction": blue_builder.getCo2CaptureFraction(),
    "capture_actual_fraction": blue_capture.getActualCaptureFraction(),
    "tube_duty_kW": blue_furnace.getTubeHeatDemandKW(),
    "heat_balance_ratio": blue_furnace.getHeatBalanceRatio(),
    "methane_conversion": blue_furnace.getTubeReformer().getMethaneConversion(),
    "ht_shift_co_conversion": blue_ht_shift.getCarbonMonoxideConversion(),
    "lt_shift_co_conversion": blue_lt_shift.getCarbonMonoxideConversion(),
    "captured_co2_kg_per_hr": blue_builder.getCapturedCo2MassFlowKgPerHour(),
    "h2_product_kg_per_hr": blue_builder.getHydrogenProductMassFlowKgPerHour(),
    "carbon_intensity_kgCO2_per_kgH2": blue_builder.getCarbonIntensityKgCO2PerKgH2(),
    "gross_carbon_intensity_kgCO2_per_kgH2": blue_builder.getGrossCarbonIntensityKgCO2PerKgH2(),
    "psa_h2_purity": blue_psa.getH2Purity(),
    "psa_h2_recovery": blue_psa.getH2Recovery(),
}
display(pd.DataFrame([blue_summary]))
print(blue_builder.getCaptureReadinessSummary())


## 5. Electrolyzer Stack, I-V Curve, and Cost Hook

The electrolysis cells demonstrate PEM, alkaline, SOEC, and AEM technology selectors through the NeqSim `Electrolyzer` class. Stack power and specific energy come from the water flow, Faradaic efficiency, and cell voltage or I-V characteristic.


In [ ]:
water = J.thermo.system.SystemSrkEos(298.15, 1.0)
water.addComponent("water", 55.5)
water.setMixingRule("classic")
water_feed = J.process.equipment.stream.Stream("demin water", water)
water_feed.setFlowRate(1000.0, "kg/hr")

el = Electrolyzer("PEM stack", water_feed)
el.setTechnology(ElectrolyzerTechnology.PEM)
iv = ElectrolyzerIVCharacteristic(ElectrolyzerTechnology.PEM)
el.setIVCharacteristic(iv)
el.setCurrentDensity(2.0)
el.run()
mech = ElectrolyzerMechanicalDesign(el)
mech.calcDesign()
el_cost = ElectrolyzerCostEstimate(mech)
el_cost.setTechnology("PEM")
{
    "cell_voltage_V": el.getCellVoltage(),
    "stack_power_MW": el.getStackPower() / 1.0e6,
    "specific_energy_kWh_per_kg_H2": el.getSpecificEnergyConsumption_kWh_per_kg_H2(),
    "hydrogen_flow_kg_hr": el.getHydrogenOutStream().getFlowRate("kg/hr"),
    "purchased_equipment_cost_USD": el_cost.getPurchasedEquipmentCost(),
}


In [ ]:
def run_electrolyzer_technology_case(label, technology, water_kg_hr=1000.0):
    case_water = J.thermo.system.SystemSrkEos(298.15, 1.0)
    case_water.addComponent("water", 55.5)
    case_water.setMixingRule("classic")
    case_feed = J.process.equipment.stream.Stream(label + " water feed", case_water)
    case_feed.setFlowRate(water_kg_hr, "kg/hr")

    case_el = Electrolyzer(label + " stack", case_feed)
    case_el.setTechnology(technology)
    case_el.setIVCharacteristic(ElectrolyzerIVCharacteristic(technology))
    case_el.run()
    return {
        "technology": label,
        "cell_voltage_V": case_el.getCellVoltage(),
        "specific_energy_kWh_per_kg_H2": case_el.getSpecificEnergyConsumption_kWh_per_kg_H2(),
        "hydrogen_flow_kg_hr": case_el.getHydrogenOutStream().getFlowRate("kg/hr"),
        "stack_power_MW": case_el.getStackPower() / 1.0e6,
    }

electrolyzer_technology_table = pd.DataFrame([
    run_electrolyzer_technology_case("PEM", ElectrolyzerTechnology.PEM),
    run_electrolyzer_technology_case("Alkaline", ElectrolyzerTechnology.ALKALINE),
    run_electrolyzer_technology_case("SOEC", ElectrolyzerTechnology.SOEC),
    run_electrolyzer_technology_case("AEM", ElectrolyzerTechnology.AEM),
])
display(electrolyzer_technology_table)
ax = electrolyzer_technology_table.plot(
    x="technology", y="specific_energy_kWh_per_kg_H2", kind="bar", legend=False
)
ax.set_xlabel("Electrolyzer technology")
ax.set_ylabel("Specific energy (kWh/kg H2)")
ax.set_title("Green hydrogen technology comparison")
plt.xticks(rotation=0)
plt.show()


## 6. Hydrogen Compression and Intercooling

This reproduces the transport notebook's structure: two compression stages, intercooling, export cooling, and MW-scale power accounting. The pressure levels and flow rate are illustrative and should be replaced by project values.


In [ ]:
h2 = J.thermo.system.SystemSrkEos(273.15 + 20.0, 90.0)
h2.addComponent("hydrogen", 1.0)
h2.setMixingRule("classic")
export_feed = J.process.equipment.stream.Stream("H2 export feed", h2)
export_feed.setFlowRate(30.0, "MSm3/day")

c1 = J.process.equipment.compressor.Compressor("C-101", export_feed)
c1.setOutletPressure(120.0, "bara")
c1.setIsentropicEfficiency(0.77)
k1 = J.process.equipment.heatexchanger.Cooler("E-101 intercooler", c1.getOutletStream())
k1.setOutTemperature(273.15 + 20.0)

c2 = J.process.equipment.compressor.Compressor("C-102", k1.getOutletStream())
c2.setOutletPressure(180.0, "bara")
c2.setIsentropicEfficiency(0.77)
k2 = J.process.equipment.heatexchanger.Cooler("E-102 export cooler", c2.getOutletStream())
k2.setOutTemperature(273.15 + 20.0)

for unit in [export_feed, c1, k1, c2, k2]:
    unit.run()

compression = pd.DataFrame([
    {"unit": "C-101", "power_MW": c1.getPower("MW"), "outlet_T_C": c1.getOutletStream().getTemperature("C")},
    {"unit": "E-101", "duty_MW": k1.getDuty("MW"), "outlet_T_C": k1.getOutletStream().getTemperature("C")},
    {"unit": "C-102", "power_MW": c2.getPower("MW"), "outlet_T_C": c2.getOutletStream().getTemperature("C")},
    {"unit": "E-102", "duty_MW": k2.getDuty("MW"), "outlet_T_C": k2.getOutletStream().getTemperature("C")},
])
display(compression)


## 7. Pipeline Transport, Profiles, and ISO 6976

    The pipeline cell uses `PipeBeggsAndBrills` for a dry-H2 export-line screening case and extracts profiles for plotting. ISO 6976 adds calorific value, Wobbe index, and relative density checks.


In [ ]:
pipe_in = k2.getOutletStream()
pipe_in.getThermoSystem().getPhase(0).getPhysicalProperties().setViscosityModel("friction theory")

pipe = J.process.equipment.pipeline.PipeBeggsAndBrills("300 km H2 pipeline", pipe_in)
pipe.setLength(300000.0)
pipe.setDiameter(1.2)
pipe.setElevation(0.0)
pipe.setPipeWallRoughness(5.0e-6)
pipe.setNumberOfIncrements(50)
pipe.setConstantSurfaceTemperature(5.0, "C")
pipe.setHeatTransferCoefficient(8.0)
pipe.run()

profile = pd.DataFrame({
    "length_km": [x / 1000.0 for x in list(pipe.getLengthProfile())],
    "pressure_bara": list(pipe.getPressureProfile()),
    "temperature_C": [t - 273.15 for t in list(pipe.getTemperatureProfile())],
})
display(profile.tail())
ax = profile.plot(x="length_km", y="pressure_bara", legend=False)
ax.set_xlabel("Length (km)")
ax.set_ylabel("Pressure (bara)")
ax.set_title("Hydrogen pipeline pressure profile")
plt.show()

iso = J.standards.gasquality.Standard_ISO6976(pipe_in.getThermoSystem(), 15.0, 15.0, "mass")
iso.calculate()
{
    "outlet_pressure_bara": pipe.getOutletStream().getPressure("bara"),
    "outlet_temperature_C": pipe.getOutletStream().getTemperature("C"),
    "superior_calorific_value_MJ_per_kg": iso.getValue("SuperiorCalorificValue") / 1000.0,
    "superior_wobbe_kWh_per_kg": iso.getValue("SuperiorWobbeIndex") / 3600.0,
    "relative_density": iso.getValue("RelativeDensity"),
}


## 8. Para-Ortho Hydrogen and Materials Screening Inputs

Cryogenic hydrogen calculations need para/ortho spin-isomer corrections. Materials work needs an operating envelope rather than a fake one-click material answer.


In [ ]:
H2Spin = ParaOrthoH2Correction
spin_rows = []
for temperature_K in [300.0, 77.0, 40.0, 20.0]:
    spin_rows.append({
        "temperature_K": temperature_K,
        "equilibrium_para_fraction": H2Spin.getEquilibriumParaFraction(temperature_K),
        "normal_to_equilibrium_heat_kJ_kg": H2Spin.getNormalToEquilibriumHeatJPerKg(temperature_K) / 1000.0,
        "cp_correction_J_kgK": H2Spin.getCpCorrectionJPerKgK(temperature_K),
        "thermal_conductivity_factor": H2Spin.getThermalConductivityCorrectionFactor(temperature_K),
    })
display(pd.DataFrame(spin_rows))

materials_screening_inputs = {
    "standards": ["ASME B31.12", "API 941", "ISO 14687", "API 521"],
    "h2_partial_pressure_bara": pipe_in.getPressure("bara"),
    "max_compressor_discharge_temperature_C": max(c1.getOutletStream().getTemperature("C"), c2.getOutletStream().getTemperature("C")),
    "min_pipeline_temperature_C": min(profile["temperature_C"]),
    "cyclic_pressure_delta_bar": 180.0 - 90.0,
    "notes": "Use these values as inputs to material, weld, fracture, leak, and consequence reviews.",
}
materials_screening_inputs


In [ ]:
results = {
    "property_reference_rows": df_ref.to_dict(orient="records"),
    "smr_equilibrium_example": run_smr_gibbs(850.0),
    "psa": {"h2_purity": psa.getH2Purity(), "h2_recovery": psa.getH2Recovery()},
    "blue_hydrogen": blue_summary,
    "electrolyzer": {"specific_energy_kWh_per_kg_H2": el.getSpecificEnergyConsumption_kWh_per_kg_H2()},
    "green_hydrogen_technology_cases": electrolyzer_technology_table.to_dict(orient="records"),
    "compression": compression.to_dict(orient="records"),
    "pipeline": {
        "outlet_pressure_bara": pipe.getOutletStream().getPressure("bara"),
        "outlet_temperature_C": pipe.getOutletStream().getTemperature("C"),
    },
    "materials_screening_inputs": materials_screening_inputs,
}
print(json.dumps(results, indent=2, default=str)[:2000])
